# VGG16 — Visualize learned image features

Use case: inspect how a CNN transforms an image into feature maps. VGG16 uses a simple stack of convolution layers. A feature map shows the response of one learned filter at different image locations.

Run the cells from top to bottom. Each notebook is independent.


## Step 1 — Install packages
Use Python 3.11 or 3.12 as a straightforward training environment, or Google Colab. Install into the active notebook kernel. Restart the kernel after the first installation if required. Internet is needed initially for packages, sample images/data and ImageNet weights. A GPU is optional; CPU runs can be slower. Model downloads can be hundreds of MB.


In [ ]:
%pip install tensorflow numpy matplotlib scikit-learn pillow


## Step 2 — Import libraries
TensorFlow runs the neural network. Keras provides the model API. NumPy handles arrays and Matplotlib displays pictures.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

keras.utils.set_random_seed(42)
print("TensorFlow:", tf.__version__)


## Step 3 — Load the example image
We keep the original visible so that we can compare it with the layer outputs.


In [ ]:
image_path = keras.utils.get_file(
    "sunflower.jpg",
    "https://storage.googleapis.com/download.tensorflow.org/example_images/592px-Red_sunflower.jpg"
)
# To use your own picture, replace image_path with a local filename.
original = keras.utils.load_img(image_path)
plt.figure(figsize=(6, 6))
plt.imshow(original)
plt.axis("off")
plt.show()


## Step 4 — Load VGG16 and select layers
Early layers tend to respond to simpler patterns; deeper responses combine more patterns. A filter is a learned array of weights, not a manually selected all-ones matrix.


In [ ]:
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
base_model = VGG16(weights="imagenet", include_top=False)
layer_names = ["block1_conv1", "block2_conv1", "block3_conv1"]
layer_outputs = []
for name in layer_names:
    layer_outputs.append(base_model.get_layer(name).output)
feature_model = keras.Model(base_model.inputs, layer_outputs)


## Step 5 — Extract feature maps
The final dimension is the number of filters. Each filter produces its own feature map.


In [ ]:
picture = keras.utils.load_img(image_path, target_size=(224, 224))
pixels = keras.utils.img_to_array(picture)
batch = preprocess_input(np.expand_dims(pixels, axis=0))
feature_maps = feature_model.predict(batch, verbose=0)
for name, maps in zip(layer_names, feature_maps):
    print(name, maps.shape)


## Step 6 — Display eight channels per layer
Bright areas indicate stronger activation in this visualization. Each channel is scaled separately by Matplotlib, so brightness should not be compared quantitatively across maps.


In [ ]:
for name, maps in zip(layer_names, feature_maps):
    plt.figure(figsize=(12, 5))
    for channel in range(8):
        plt.subplot(2, 4, channel + 1)
        plt.imshow(maps[0, :, :, channel], cmap="viridis")
        plt.title(f"Channel {channel}")
        plt.axis("off")
    plt.suptitle(name)
    plt.tight_layout()
    plt.show()


## Understand the result
These are activations, not the filter weights themselves. A channel is not necessarily a human-readable detector such as “flower” or “eye.” We inspect a pretrained network without changing its weights.


## Practice
1. Change the input image or selected image index.
2. Explain the output in your own words.
3. Compare the result with the original image and record one limitation.


## References
- [Keras Applications](https://keras.io/api/applications/)
- [CIFAR-10 dataset](https://www.cs.toronto.edu/~kriz/cifar.html)
- [Keras Grad-CAM example](https://keras.io/examples/vision/grad_cam/)
